# CONFIG

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/BIOM/notebooks
/home/turbotowerlnx/Documents/Master/BIOM/app


In [2]:
from src.config import Configuration    

CONFIG = Configuration()

# GET ALL POSSIBLE HAAR

In [3]:
from typing import List
from src.model import Feature, Rectangle


def generate_all_features(win_w: int = 24, win_h: int = 24) -> List[Feature]:
    features = []
    
    for w in range(1, win_w + 1):
        for h in range(1, win_h + 1):
            for x in range(win_w - w + 1):
                for y in range(win_h - h + 1):
                    
                    # 2-rectangle horizontal (Left/Right)
                    if w % 2 == 0:
                        w_half = w // 2
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w_half, h, -1.0),
                            Rectangle(x + w_half, y, w_half, h, 1.0)
                        ]))
                        
                    # 2-Rectangle vertical (Top/Bottom)
                    if h % 2 == 0:
                        h_half = h // 2
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w, h_half, -1.0),
                            Rectangle(x, y + h_half, w, h_half, 1.0)
                        ]))
                        
                    # 3-Rectangle horizontal (Left/Center/Right)
                    if w % 3 == 0:
                        w_third = w // 3
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w_third, h, -1.0),
                            Rectangle(x + w_third, y, w_third, h, 2.0), # Center weight compensates for 2 outside Rectangles
                            Rectangle(x + 2 * w_third, y, w_third, h, -1.0)
                        ]))
                        
                    # 3-Rectangle vertical (Top/Center/Bottom)
                    if h % 3 == 0:
                        h_third = h // 3
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w, h_third, -1.0),
                            Rectangle(x, y + h_third, w, h_third, 2.0),
                            Rectangle(x, y + 2 * h_third, w, h_third, -1.0)
                        ]))
                        
                    # 4-Rectangle (Checkerboard)
                    if w % 2 == 0 and h % 2 == 0:
                        w_half, h_half = w // 2, h // 2
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w_half, h_half, 1.0),
                            Rectangle(x + w_half, y, w_half, h_half, -1.0),
                            Rectangle(x, y + h_half, w_half, h_half, -1.0),
                            Rectangle(x + w_half, y + h_half, w_half, h_half, 1.0)
                        ]))
                        
    return features

In [4]:
all_features = generate_all_features()

# COMPUTE ALL FEATURES FOR DATASET

In [5]:
from maikol_utils.file_utils import list_dir_files

faces, n = list_dir_files(CONFIG.faces_path, recursive=True)
print(f"Found {n} files in {CONFIG.faces_path}")

non_faces, n = list_dir_files(CONFIG.crops_path, recursive=True)
print(f"Found {n} files in {CONFIG.crops_path}")

Found 136965 files in ../data/ViolaJones/face_images
Found 280000 files in ../data/ViolaJones/crops


In [7]:
import os
import cv2
import numpy as np
import multiprocessing as mp
from tqdm import tqdm
from src.data import get_integral_image, get_integral_squared_image, get_std_from_integral_images
from src.model import compute_feature

def _extract_features_from_face_path(face_path):
    img = cv2.imread(face_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    integral_img = get_integral_image(img)
    integral_img_2 = get_integral_squared_image(img)

    # Calculate std
    H, W = img.shape
    r1 = np.array([[0]])
    r2 = np.array([[H - 1]])
    c1 = np.array([[0]])
    c2 = np.array([[W - 1]])

    _, std_dev = get_std_from_integral_images(integral_img, integral_img_2, r1, r2, c1, c2)
    std_dev = std_dev[0, 0]

    if std_dev <= 0:
        std_dev = 1.0

    features = [compute_feature(integral_img, feature) * std_dev for feature in all_features]
    return np.asarray(features, dtype=np.float32)

n_workers = int(max(1, (os.cpu_count() or 1) - 1) * 4 / 5)
chunksize = 8

with mp.get_context("fork").Pool(processes=n_workers) as pool:
    results = list(
        tqdm(
            pool.imap(_extract_features_from_face_path, faces, chunksize=chunksize),
            total=len(faces),
            desc=f"Extracting face features ({n_workers} workers)",
        )
    )

valid_results = [r for r in results if r is not None]
if valid_results:
    X_train_faces = np.stack(valid_results, axis=0).astype(np.float32, copy=False)
else:
    X_train_faces = np.empty((0, len(all_features)), dtype=np.float32)

dropped = len(results) - len(valid_results)
if dropped:
    print(f"Skipped {dropped} unreadable images.")
print(f"Computed face features for {X_train_faces.shape[0]} images.")
print(f"X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")

Extracting face features (18 workers):  12%|█▏        | 16816/136965 [10:08<1:12:24, 27.66it/s]Process ForkPoolWorker-18:
Process ForkPoolWorker-5:
Process ForkPoolWorker-12:
Process ForkPoolWorker-17:
Process ForkPoolWorker-14:
Process ForkPoolWorker-7:
Process ForkPoolWorker-3:
Process ForkPoolWorker-10:
Process ForkPoolWorker-15:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkPoolWorker-8:
Process ForkPoolWorker-9:
Process ForkPoolWorker-4:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkPoolWorker-16:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Process ForkPoolWorker-6:
Process ForkPoolWorker-2:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
 

KeyboardInterrupt: 

# TRAIN ADA-BOOSTS

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# X_train shape: (num_images, num_features) - precalculated feature values
# y_train shape: (num_images,) - 1 for face, 0 for background

def train_stage_for_tpr(X_train, y_train, n_features, target_tpr=0.995):
    # 1. Train standard AdaBoost using Decision Stumps
    clf = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n_features,
        algorithm='SAMME' # Standard discrete AdaBoost
    )
    clf.fit(X_train, y_train)

    # 2. Get continuous scores for all actual faces
    face_scores = clf.decision_function(X_train[y_train == 1])
    
    # 3. Sort scores and find the exact threshold to achieve target_tpr
    # If we have 1000 faces and target_tpr is 0.995, we can only drop the bottom 5.
    face_scores_sorted = np.sort(face_scores)
    drop_count = int(len(face_scores) * (1.0 - target_tpr))
    
    custom_threshold = face_scores_sorted[drop_count]

    return clf, custom_threshold

# To predict later: 
# passes_stage = clf.decision_function(X_test) >= custom_threshold